# Gate C: the public dataset reproduces the headline

Runs on a free Colab **CPU** runtime in a few minutes, with no token: the dataset is public.

Two checks:
1. **Rerun.** Download the raw run files from [Faruna01/persona-collapse-runs](https://huggingface.co/datasets/Faruna01/persona-collapse-runs), rerun `scripts/analyze.py`, and compare with the tables committed in `results/`.
2. **Independent recompute.** Load the flat `turns` table with `datasets.load_dataset` and recompute every mean score from scratch.

Then it redraws the headline chart: drift from each language's own coding baseline, by topic.

In [ ]:
!git clone -q https://github.com/farunag/persona-collapse.git
%cd persona-collapse
!pip -q install datasets huggingface_hub pandas matplotlib

In [ ]:
import os
from huggingface_hub import hf_hub_download

REPO = "Faruna01/persona-collapse-runs"
RAW = "/content/runs"
for model, runs in [("qwen", ["uncapped", "capped"]), ("gemma", ["uncapped"])]:
    for run in runs:
        for ext in [".jsonl", ".proj.jsonl", ".langid.jsonl"]:
            hf_hub_download(REPO, f"{model}/replay/{model}_{run}{ext}", repo_type="dataset", local_dir=RAW)
print(sorted(os.listdir(f"{RAW}/qwen/replay")), sorted(os.listdir(f"{RAW}/gemma/replay")))

In [ ]:
import subprocess, sys

OUT = "/content/gate_c"
os.makedirs(OUT, exist_ok=True)
stems = [f"{RAW}/{m}/replay/{m}_{r}" for m, r in [("gemma", "uncapped"), ("qwen", "uncapped"), ("qwen", "capped")]]
cmd = [sys.executable, "scripts/analyze.py",
       "--proj", *[s + ".proj.jsonl" for s in stems],
       "--langid", *[s + ".langid.jsonl" for s in stems],
       "--replay", *[s + ".jsonl" for s in stems],
       "--out", OUT, "--overwrite"]
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout[-2000:], res.stderr[-2000:])

In [ ]:
import pandas as pd

KEY = ["model", "capped", "lang", "domain"]
COLS = ["mean_score", "baseline", "drift", "drift_lo", "drift_hi"]
repo = pd.read_csv("results/stage2_domain_drift.csv")
rerun = pd.read_csv(f"{OUT}/stage2_domain_drift.csv")
m = repo.merge(rerun, on=KEY + ["n_convs", "turns"], suffixes=("_repo", "_rerun"))
diff = max((m[c + "_repo"] - m[c + "_rerun"]).abs().max() for c in COLS)
same_nan = all((m[c + "_repo"].isna() == m[c + "_rerun"].isna()).all() for c in COLS)
print(f"rows: repo {len(repo)}, rerun {len(rerun)}, matched {len(m)}; max abs difference {diff:.2e}")
check1 = len(m) == len(repo) == len(rerun) and same_nan and diff < 1e-9
print("CHECK 1 PASS: the rerun matches results/stage2_domain_drift.csv" if check1 else "CHECK 1 FAIL")

In [ ]:
from datasets import load_dataset

df = load_dataset(REPO, "turns", split="train").to_pandas()
df["model_short"] = df.model.str.split("/").str[-1]
window = {(r.model, bool(r.capped), r.domain): tuple(map(int, r.turns.split("-"))) for r in repo.itertuples()}
rows = []
for (mod, cap, lang, dom), g in df.groupby(["model_short", "capped", "language", "domain"]):
    lo, hi = window[(mod, bool(cap), dom)]
    g = g[(g.turn >= lo) & (g.turn <= hi)]
    rows.append({"model": mod, "capped": bool(cap), "lang": lang, "domain": dom,
                 "n_flat": g.conv_id.nunique(), "mean_flat": g.groupby("conv_id").score.mean().mean()})
flat = pd.DataFrame(rows)
chk = repo.merge(flat, on=KEY)
d2 = (chk.mean_score - chk.mean_flat).abs().max()
check2 = len(chk) == len(repo) and (chk.n_convs == chk.n_flat).all() and d2 < 1e-9
print(f"{len(df)} turn rows; matched {len(chk)} of {len(repo)} cells; max abs difference {d2:.2e}")
print("CHECK 2 PASS: the flat table reproduces every mean score" if check2 else "CHECK 2 FAIL")

In [ ]:
import matplotlib.pyplot as plt

COLORS = {"en": "#2a78d6", "ha": "#eb6834", "yo": "#1baf7a"}
NAMES = {"en": "English", "ha": "Hausa", "yo": "Yoruba"}
d = rerun[(~rerun.capped) & (rerun.domain != "coding")]
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), sharey=True)
for ax, mod in zip(axes, ["Qwen3-32B", "gemma-2-27b-it"]):
    s = d[d.model == mod]
    for i, dom in enumerate(["therapy", "metacognitive"]):
        for j, l in enumerate(["en", "ha", "yo"]):
            r = s[(s.domain == dom) & (s.lang == l)].iloc[0]
            ax.errorbar(i + (j - 1) * 0.22, r.drift, yerr=[[r.drift - r.drift_lo], [r.drift_hi - r.drift]],
                        fmt="o", color=COLORS[l], capsize=3, label=NAMES[l] if i == 0 else None)
    ax.axhline(0, color="#999", lw=0.8)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Therapy", "Self-reflection"]); ax.set_title(mod)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("Drift from own-language coding baseline")
axes[0].legend(frameon=False)
fig.suptitle("Uncapped drift by topic and language (95% CI)", x=0.02, ha="left")
plt.tight_layout(); plt.show()
print("GATE C PASS" if check1 and check2 else "GATE C FAIL: see the checks above")

**Reading the chart.** Negative drift = further from the model's default Assistant than in that language's coding conversations. Qwen drifts far more in English than in Hausa or Yoruba, but the sensitivity check (`results/sensitivity_report.md`, `results/role_adoption_report.md`) shows the axis itself responds about 3× more weakly in Hausa and Yoruba, so smaller Hausa/Yoruba drift is not evidence of a steadier model. Gemma's Hausa/Yoruba values mix persona with reply language.